In [17]:
from pathlib import Path 
from PIL import Image 
import pandas as pd 
import hashlib
from collections import Counter 

# Dataset locations 
DATASET_DIR = Path("../dataset/raw/Plant Village Dataset")

# Verify the path 
print("Dataset path:",DATASET_DIR.resolve())
print("Dataset exists:", DATASET_DIR.exists())

# Supported image formats
SUPPORTED_FORMATS = {".jpg",".jpeg",".png"}

#Dataset splits 
SPLITS = ["Train","Val","Test"]


Dataset path: A:\Crop-Disease-Detection\dataset\Raw\Plant Village Dataset
Dataset exists: True


In [18]:
image_records = []
unusual_files = []
empty_classes = []

# Check each dataset split
for split in SPLITS:

    split_path = DATASET_DIR / split

    if not split_path.exists():
        print(f"Warning: {split} folder not found!")
        continue

    # Check each disease class
    for class_folder in split_path.iterdir():

        if not class_folder.is_dir():
            continue

        # Find all files inside this class folder
        files = [f for f in class_folder.rglob("*") if f.is_file()]

        supported_files = [
            f for f in files
            if f.suffix.lower() in SUPPORTED_FORMATS
        ]

        if not supported_files:
            empty_classes.append({
                "Dataset": split,
                "Class": class_folder.name
            })

        for file_path in files:

            extension = file_path.suffix.lower()

            # Report unusual file formats
            if extension not in SUPPORTED_FORMATS:
                unusual_files.append({
                    "Dataset": split,
                    "Class": class_folder.name,
                    "File": str(file_path),
                    "Extension": extension or "[no extension]"
                })
                continue

            record = {
                "Dataset": split,
                "Class": class_folder.name,
                "File": str(file_path),
                "Extension": extension,
                "Width": None,
                "Height": None,
                "Mode": None,
                "Format": None,
                "Size_KB": round(file_path.stat().st_size / 1024, 2),
                "Status": "Valid",
                "Error": None,
                "SHA256": None
            }

            try:
                # First check whether the image is valid
                with Image.open(file_path) as img:
                    img.verify()

                # Reopen it to read image properties
                with Image.open(file_path) as img:
                    record["Width"] = img.width
                    record["Height"] = img.height
                    record["Mode"] = img.mode
                    record["Format"] = img.format

                # Calculate exact file hash
                file_hash = hashlib.sha256(
                    file_path.read_bytes()
                ).hexdigest()

                record["SHA256"] = file_hash

            except Exception as e:
                record["Status"] = "Corrupted"
                record["Error"] = str(e)

            image_records.append(record)

print("Image scanning completed!")

Image scanning completed!


In [19]:
# Convert results into DataFrames

quality_df = pd.DataFrame(image_records)
unusual_df= pd.DataFrame(unusual_files)
empty_df=pd.DataFrame(empty_classes)

print("Total Supported Images: ",len(quality_df))

print(
    "Valid Images: ",
    (quality_df["Status"] == "Valid").sum()
)

print(
    "Corrupted Images: ",
    (quality_df["Status"] == "Corruped").sum()
)

print("Unusual files: ",len(unusual_df))
print("Empty class folders: ",len(empty_df))

Total Supported Images:  67118
Valid Images:  67118
Corrupted Images:  0
Unusual files:  0
Empty class folders:  0


In [20]:
# Detect duplicate images

# Consider only valid images 
valid_df = quality_df[
    quality_df["Status"] == "Valid"
].copy()

# Find hashes that appears more than once 
duplicate_df = valid_df[
    valid_df.duplicated(subset="SHA256", keep=False)
].sort_values("SHA256")

print("Total duplicate file enteries: ",len(duplicate_df))

duplicate_df[
    ["Dataset", "Class", "File","SHA256"]
]


Total duplicate file enteries:  66


,Dataset,Class,File,SHA256
47157,Train,Tomato - Healthy,..\dataset\raw\Plant Village Dataset\Train\Tom...,12852b3616bcf0ba39b03b5677e74f50f5f6abf4d38957...
47697,Train,Tomato - Healthy,..\dataset\raw\Plant Village Dataset\Train\Tom...,12852b3616bcf0ba39b03b5677e74f50f5f6abf4d38957...
48679,Train,Tomato - Late Blight,..\dataset\raw\Plant Village Dataset\Train\Tom...,1d202e8c6b21351c851a4220c6ac5be10c4bf7f659fa82...
48915,Train,Tomato - Late Blight,..\dataset\raw\Plant Village Dataset\Train\Tom...,1d202e8c6b21351c851a4220c6ac5be10c4bf7f659fa82...
51731,Train,Tomato - Septoria Leaf Spot,..\dataset\raw\Plant Village Dataset\Train\Tom...,257c75912bb03678b9af067545101252aa59790950d9ad...
...,...,...,...,...
5897,Train,Apple - Healthy,..\dataset\raw\Plant Village Dataset\Train\App...,ec2fe5801c82d7ffff332b9429480c6a432830ccce2f7e...
7470,Train,Apple - Healthy,..\dataset\raw\Plant Village Dataset\Train\App...,f9658c2a55775a239cd5fecc3a42114ec84f46cd68aa73...
7287,Train,Apple - Healthy,..\dataset\raw\Plant Village Dataset\Train\App...,f9658c2a55775a239cd5fecc3a42114ec84f46cd68aa73...
7162,Train,Apple - Healthy,..\dataset\raw\Plant Village Dataset\Train\App...,ffd85543367c06fcc7008315354b387dc9f995627a900c...


In [21]:
# Analyze Image Dimensions

dimension_counts= (
    valid_df.groupby(["Width","Height"])
    .size()
    .reset_index(name="Image_Count")
    .sort_values("Image_Count",ascending=False)
)

dimension_counts.head(20)


,Width,Height,Image_Count
0,256,256,67118


In [22]:
# To check smallest and largest images
print("Smallest width:", valid_df["Width"].min())
print("Largest width:", valid_df["Width"].max())

print("Smallest height:", valid_df["Height"].min())
print("Largest height:", valid_df["Height"].max())

Smallest width: 256
Largest width: 256
Smallest height: 256
Largest height: 256


In [23]:
# Analyze color models

mode_counts = valid_df["Mode"].value_counts()

print(mode_counts)

Mode
RGB    67118
Name: count, dtype: int64


In [24]:
# Check corrupted images and empty folders 

# Display corrupted images
corrupted_df = quality_df[
    quality_df["Status"] == "Corrupted"
]
corrupted_df[["Dataset","Class","File","Error"]]


,Dataset,Class,File,Error


In [31]:
# Find duplicate images across different dataset splits

cross_split_duplicates = (
    quality_df.groupby("SHA256")
    .filter(
        lambda group:
        group["Dataset"].nunique() > 1
        and group["SHA256"].notna().all()
    )
    .sort_values("SHA256")
    .reset_index(drop=True)
)

print("Cross-split duplicates found:", len(cross_split_duplicates))

Cross-split duplicates found: 14


In [32]:
# To check the folders that contains no valid images

valid_class_counts = (
    valid_df.groupby(["Dataset", "Class"])
    .size()
)

for split in SPLITS:

    split_path = DATASET_DIR / split

    if not split_path.exists():
        continue

    for folder in split_path.iterdir():

        if not folder.is_dir():
            continue

        count = valid_class_counts.get(
            (split, folder.name), 0
        )

        if count == 0:
            print(
                f"No valid images: {split}/{folder.name}"
            )


In [33]:
# Save all quality reports 

# Create report folder
REPORT_DIR = Path("../dataset/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save reports
quality_df.to_csv(
    REPORT_DIR / "image_quality_report.csv",
    index=False
)

duplicate_df.to_csv(
    REPORT_DIR / "duplicate_images.csv",
    index=False
)

cross_split_duplicates.to_csv(
    REPORT_DIR / "cross_split_duplicates.csv",
    index=False
)

unusual_df.to_csv(
    REPORT_DIR / "unusual_files.csv",
    index=False
)

empty_df.to_csv(
    REPORT_DIR / "empty_classes.csv",
    index=False
)

dimension_counts.to_csv(
    REPORT_DIR / "image_dimensions.csv",
    index=False
)

print("All quality reports saved successfully!")

All quality reports saved successfully!
